# 00 — Setup Colab (M1)
Notebook khởi tạo môi trường. Chạy trước tất cả notebook khác.

**Bước:** chọn `SOURCE` ("drive" hoặc "git") → Runtime → GPU (T4) → Run all.

In [ ]:
# === Cell setup linh hoat: Google Drive HOAC git clone ===
SOURCE = "git"  # "drive" hoac "git"
GIT_URL = "https://github.com/<user>/Master-UIT-MedSignal.git"  # TODO: doi thanh repo cua nhom
DRIVE_PATH = "/content/drive/MyDrive/Master-UIT-MedSignal"      # TODO: vi tri tren Drive

import os, sys
if SOURCE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = DRIVE_PATH
elif SOURCE == "git":
    PROJECT = "/content/Master-UIT-MedSignal"
    if not os.path.exists(PROJECT):
        os.system(f"git clone {GIT_URL} {PROJECT}")
else:
    raise ValueError("SOURCE phai la 'drive' hoac 'git'")

os.chdir(PROJECT)
sys.path.append(PROJECT)
print("PROJECT =", PROJECT)
print(os.listdir(PROJECT))

In [ ]:
# Cai dependencies
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# === Smoke test: kiem tra du lieu thuc te + load 1 batch ===
from src.data import preprocess as P
from src.data.dataset import CarotidDataset, collate_fn
from src.data.splits import stratified_folds, fold_summary
from torch.utils.data import DataLoader

cfg = P.load_config("configs/config.yaml")
df = P.load_dataframe(cfg)
print("So ca:", len(df))
print("Plaque_present:\n", df[cfg['columns']['target_plaque']].value_counts())
print("Echogenicity:\n", df[cfg['columns']['target_echo']].value_counts())

folds = stratified_folds(df, cfg)
print(fold_summary(df, folds, cfg))

In [ ]:
# Load thu 1 batch -> kiem tra shape khop data-contract
tr, va = folds[0]
df_tr = df.iloc[tr]
scaler = P.fit_scaler(P.encode_categorical(df_tr, cfg), cfg)
ds = CarotidDataset(df_tr, cfg, scaler)
dl = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate_fn)

batch = next(iter(dl))
print("tabular :", batch['tabular'].shape)      # [8, 9]
print("imt_img :", batch['imt_img'].shape)      # [8, 1, 256, 256]
print("cca_imgs:", batch['cca_imgs'].shape)     # [8, 4, 1, 256, 256]
print("cca_mask:", batch['cca_mask'].shape)     # [8, 4]
print("labels  :", {k: v.shape for k, v in batch['labels'].items()})
print("\n✅ Pipeline OK — san sang train.")